# Module 2: RAG code generator (Groq)
Pipeline:

`user request -> intent -> retrieve relevant snippets -> Groq LLM -> code`

- **Retriever:** local embeddings (`sentence-transformers`) + cosine similarity. No vector DB needed at this scale.
- **Generator:** an open model served by **Groq**. Retrieved snippets go into the prompt so answers are grounded in *your* knowledge base.
- **Knowledge base:** any `.ipynb`, `.py`, `.md` or `.txt` files in a `knowledge_base/` folder next to this notebook.

In [ ]:
# run once, then restart the kernel if needed
# %pip install groq sentence-transformers numpy

---
**1. API key and model**  
Create a free key at [console.groq.com](https://console.groq.com), then set it **before launching Jupyter**:
```bash
export GROQ_API_KEY='****'
```
Never paste the key into the notebook or commit it to GitHub. If the variable isn't set, the cell below asks for it securely.

Groq retires and adds models from time to time. If you get a "model not found" error, pick a current name from the [models page](https://console.groq.com/docs/models) and change `MODEL`.

In [ ]:
import os
import getpass
from groq import Groq

key = getpass.getpass("Enter your Groq API key: ").strip()

client = Groq(api_key=key)

MODEL = 'openai/gpt-oss-120b'

In [ ]:
print(client.models.list())

---
**2. Load and chunk the knowledge base**  
- `.py` files are split into one chunk per top-level function/class (so each chunk is a complete unit of code).  
- `.md` / `.txt` files are split into paragraph groups of about 800 characters.  

In [ ]:

import json

def chunk_notebook(text, filename):
    notebook = json.loads(text)
    chunks = []

    for cell in notebook.get("cells", []):
        if cell["cell_type"] in ("code", "markdown"):
            content = "".join(cell.get("source", []))
            if content.strip():
                chunks.append({
                    "source": filename,
                    "text": content
                })

    return chunks


def load_knowledge_base(folder=KB_DIR):
    chunks = []

    for path in sorted(Path(folder).rglob("*")):
        if not path.is_file():
            continue

        text = path.read_text(encoding="utf-8", errors="ignore")

        if path.suffix == ".ipynb":
            chunks += chunk_notebook(text, path.name)
        elif path.suffix == ".py":
            chunks += chunk_python(text, path.name)
        elif path.suffix in (".md", ".txt"):
            chunks += chunk_text(text, path.name)

    return chunks


chunks = load_knowledge_base()
print(f"{len(chunks)} chunks loaded")


---
**3. Embed and retrieve**  
Each chunk becomes a vector. A query is embedded the same way, and we take the `k` chunks with the highest cosine similarity (vectors are normalised, so a dot product is enough). Retrieval runs fully locally and is free.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')

def build_index(chunks):
    return embedder.encode([c['text'] for c in chunks], normalize_embeddings=True)

chunk_vecs = build_index(chunks)
print(chunk_vecs.shape)


def retrieve(query, k=3, min_score=0.2):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = chunk_vecs @ q
    top = np.argsort(-scores)[:k]
    return [(chunks[i], float(scores[i])) for i in top if scores[i] >= min_score]

In [ ]:
print("Number of chunks:", len(chunks))
print("chunk_vecs shape:", chunk_vecs.shape)
print("query embedding shape:", embedder.encode(["test"], normalize_embeddings=True)[0].shape)

In [ ]:
# sanity check retrieval before calling the API
for c, s in retrieve('how do I read a json file'):
    print(f'{s:.2f}  {c["source"]}')

---
**4. Intent routing (hook for Module 1)**  
`classify_intent` is a **placeholder** using keywords. The Module 1 model was trained on SNIPS intents (PlayMusic, GetWeather, ...), which aren't coding intents, so it can't be dropped in directly.

In [ ]:
INTENT_INSTRUCTIONS = {
    'write_code':   'Write the requested code. Return a complete, runnable solution in a single code block, then a 1-2 sentence explanation.',
    'explain_code': 'Explain the code or concept clearly and concisely, using short examples where useful.',
    'debug_code':   'Find the bug, show the corrected code, and explain what was wrong in 1-3 sentences.',
}

def classify_intent(text):
    # TODO: replace with Module 1 prediction once it is trained on coding intents
    t = text.lower()
    if any(w in t for w in ('error', 'bug', 'fix', 'traceback', 'not working')):
        return 'debug_code'
    if any(w in t for w in ('explain', 'what does', 'how does', 'why')):
        return 'explain_code'
    return 'write_code'

---
**5. Generate with Groq**  
The retrieved chunks go into the prompt as reference material. 

In [ ]:
import time

SYSTEM = '''You are CodeBud, a coding assistant.
You are given REFERENCE snippets retrieved from the user's knowledge base.
- Prefer the reference snippets and follow any style rules in them.
- If the references don't cover the request, answer from general knowledge and say so briefly.
- Never invent library functions that you are not sure exist.'''


def build_prompt(query, intent, hits):
    if hits:
        refs = '\n\n'.join(f'[{c["source"]}]\n{c["text"]}' for c, _ in hits)
    else:
        refs = '(no relevant references found)'
    return f'''REFERENCES:
{refs}

TASK TYPE: {intent}
INSTRUCTIONS: {INTENT_INSTRUCTIONS.get(intent, INTENT_INSTRUCTIONS['write_code'])}

USER REQUEST:
{query}'''


def codebud(query, k=3, show_sources=True, retries=3):
    intent = classify_intent(query)
    hits = retrieve(query, k=k)
    prompt = build_prompt(query, intent, hits)

    resp = None
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                max_tokens=1500,
                temperature=0.2,
                messages=[
                    {'role': 'system', 'content': SYSTEM},
                    {'role': 'user', 'content': prompt},
                ],
            )
            break
        except groq.RateLimitError:
            if attempt == retries - 1:
                return 'Rate limit hit. Wait a minute and try again.'
            time.sleep(5 * (attempt + 1))
        except groq.APIError as e:
            return f'API error: {e}'

    answer = resp.choices[0].message.content
    if show_sources:
        srcs = ', '.join(f'{c["source"]} ({s:.2f})' for c, s in hits) or 'none'
        print(f'[intent: {intent}] [sources: {srcs}]\n')
    return answer

In [ ]:
print(codebud('Write a function that reads a JSON file and returns only the keys at the top level'))

In [ ]:
print(codebud('Write a function to check whether a list is sorted, in the CodeBud style'))